# Detecció d'opinions
La primera part de la pràctica 2 consta de la detecció d'opinions classificra-les coma  negatives o positives
El primer que s'ha realitzat es importar les dades que utilitzarem com a corpus

In [1]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews as mr


[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


## Algoritmes d'aprenentatge supervisat

### Preparació de les dades

La primera part de la preparació de les dades consisteix a netejar-les. Aquest procés inclou dos passos principals:

Primer, normalitzem el text posant-lo tot en minúscules i eliminant qualsevol caràcter que no sigui una lletra de l'alfabet (a-z), com ara números, signes de puntuació o símbols especials. Això fara que el model tracti "genial!", "genial," com paraules diferents.

Segon, lematitzem el text, és a dir, reduïm cada paraula a la seva forma base o lema. Per exemple, "running", "runs" i "ran" es converteixen totes en "run". Gràcies a això, el vocabulari és molt més compacte i el model pot aprendre millor les relacions entre paraules.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

#no treure les negacions, nomes les paraules amb significat -> dont, not

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text_string(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 1
    ]

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


El corpus "movie_review" ja compta amb les etiquetes "pos" i "neg" per a opinions positives i negatives respectivament. Per preparar les dades, hem separat aquestes etiquetes i hem dividit el corpus en train i test, amb un 20% del corpus per al test i un 80% per a l'entrenament dels models.
A més, s'aplica la funció feta anteriorment, que deixa el text net per poder entrenar els models adequadament.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Separar les etiquetes ("pos" i "neg") que fan referència a la categoria
documents = []
labels = []

for categoria in mr.categories():
    for fileid in mr.fileids(categoria):
        raw_text = mr.raw(fileid)                 
        clean_text = clean_text_string(raw_text)  

        if clean_text:                            
            documents.append(clean_text)
            labels.append(categoria)

# Si no fem això a vegades s'afageixen opinions buides i genera errors, 
# per tant, només si hi ha text s'afageix a les dades

# Divisió del corpus train/test

X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42
)


A continuació, com que les dades són strings, les convertim a números, ja que els models que hem escollit per classificar només funcionen amb valors numèrics. Per fer això, utilitzem CountVectorizer, que converteix el text en vectors numèrics.

Hem d'escollir l'hiperparàmetre max_features, que fa referència a la mida del vocabulari que utilitzarà el model. Per escollir-lo, cal tenir en compte que si és massa petit es perdrà molta informació, i si és massa gran augmentarà considerablement el cost computacional.

### SVM (Super Vector Machine)

Per entrenar el primer model escollit s'ha fet un GridSearch on s'avaluen diferents combinacions d'hiperparàmetres, tant del propi SVM, com ara el kernel i el paràmetre C, com del pas de vectorització, concretament el max_features. D'aquesta manera, s'escollirà el millor conjunt d'hiperparàmetres en funció de l'accuracy i altres mètriques, que és la mètrica que s'està utilitzant, i amb aquest conjunt òptim s'entrenarà el model final.

IMPORTANT:

(1, 2) significa que mires paraules individuals i també parelles de paraules consecutives (bigrames):

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Pipeline: vectorització + classificació
pipe = Pipeline([
    ("vect", CountVectorizer()),
    ("svc", SVC())
])

param_grid = {
    "vect__max_features": [100, 5000, 12000, 20000],
    "svc__kernel": ["linear", "rbf", "poly"],
    "svc__C": [0.001, 0.1, 1, 10],
    "svc__gamma": ["scale", "auto"]
}

# Cerca exhaustiva per validació creuada
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Millors paràmetres:", grid.best_params_)
print(f"Millor accuracy CV: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nAccuracy test: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Millors paràmetres: {'svc__C': 10, 'svc__gamma': 'auto', 'svc__kernel': 'rbf', 'vect__max_features': 12000}
Millor accuracy CV: 0.8275

Accuracy test: 0.8125

Classification report:

              precision    recall  f1-score   support

         neg       0.80      0.84      0.82       199
         pos       0.83      0.79      0.81       201

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400



Millors paràmetres: {'svc__C': 10, 'svc__gamma': 'auto', 'svc__kernel': 'rbf', 'vect__max_features': 12000}


### XGBoost

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# XGBoost requereix classes numèriques (0/1)
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

# Pipeline: vectorització + classificació
pipe = Pipeline([
    ("vect", CountVectorizer()),
    ("xgb", XGBClassifier(eval_metric="logloss", verbosity=0))
])

param_grid = {
    "vect__max_features": [100, 5000, 12000, 20000],
    "xgb__n_estimators": [50, 100, 300],
    "xgb__max_depth": [3, 6, 9],
    "xgb__learning_rate": [0.01, 0.05, 0.1],
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train_enc)

print("Millors paràmetres:", grid.best_params_)
print(f"Millor accuracy CV: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred_enc = best_model.predict(X_test)
y_pred = label_encoder.inverse_transform(y_pred_enc)

print(f"\nAccuracy test: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Millors paràmetres: {'vect__max_features': 12000, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 3, 'xgb__n_estimators': 300}
Millor accuracy CV: 0.8119

Accuracy test: 0.8500

Classification report:

              precision    recall  f1-score   support

         neg       0.83      0.87      0.85       199
         pos       0.87      0.83      0.85       201

    accuracy                           0.85       400
   macro avg       0.85      0.85      0.85       400
weighted avg       0.85      0.85      0.85       400



Millors paràmetres: {'vect__max_features': 12000, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 3, 'xgb__n_estimators': 300}


### Regressió Logística

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Pipeline: vectorització + classificació
pipe = Pipeline([
    ("vect", CountVectorizer()),
    ("logreg", LogisticRegression(max_iter=3000))
])

# Grid de paràmetres per a Regressió Logística
param_grid = [
    {
        "vect__max_features": [100, 5000, 12000, 20000],
        "logreg__penalty": ["l1", "l2"],
        "logreg__C": [0.01, 0.1, 1, 10]
    }
]

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Millors paràmetres:", grid.best_params_)
print(f"Millor accuracy CV: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nAccuracy test: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 32 candidates, totalling 160 fits


c:\Users\ashve\IA\q4\PLH\Lab\venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
80 fits failed out of a total of 160.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
80 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\ashve\IA\q4\PLH\Lab\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\ashve\IA\q4\PLH\Lab\venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ashve\IA\q4\PLH\Lab\venv\Lib\site-packages\sklearn\pipeline.py", line 621,

Millors paràmetres: {'logreg__C': 0.01, 'logreg__penalty': 'l2', 'vect__max_features': 5000}
Millor accuracy CV: 0.8450

Accuracy test: 0.8300

Classification report:

              precision    recall  f1-score   support

         neg       0.82      0.85      0.83       199
         pos       0.84      0.81      0.83       201

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400



c:\Users\ashve\IA\q4\PLH\Lab\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Millors paràmetres: {'logreg__C': 0.01, 'logreg__penalty': 'l2', 'vect__max_features': 5000}
